In [5]:
import os
import json
import pandas as pd
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

OCR_DIR = "./data/3_ocr_results"
SEG_DIR = "./data/5_telop_segments"
OUT_DIR = "./data/6_telop_instances"
FPS = 10


def _loads(x):
    return json.loads(x) if isinstance(x, str) else x


def assign_kdenlive_tracks(instances):
    """greedy: 시작시각 순으로 비어있는 트랙에 할당"""
    instances.sort(key=lambda x: (x["start_sec"], x["end_sec"]))
    track_end = []  # track_end[i] = i번 트랙의 마지막 end_sec
    for inst in instances:
        placed = False
        for ti, end in enumerate(track_end):
            if inst["start_sec"] >= end:
                inst["kdenlive_track"] = ti
                track_end[ti] = inst["end_sec"]
                placed = True
                break
        if not placed:
            inst["kdenlive_track"] = len(track_end)
            track_end.append(inst["end_sec"])
    return instances, len(track_end)

def process_one(ocr_pq, seg_pq, out_json, video_name):
    df_ocr = pd.read_parquet(ocr_pq).set_index("frame_num")
    df_seg = pd.read_parquet(seg_pq).set_index("frame_num")

    tracks = defaultdict(lambda: {"frames": set(), "text_scores": defaultdict(float)})

    for fnum, ocr_row in df_ocr.iterrows():
        if fnum not in df_seg.index:
            continue
        texts = _loads(ocr_row["ocr_texts"])
        scores = _loads(ocr_row["ocr_scores"])
        tids = _loads(df_seg.loc[fnum, "track_id"])

        for text, score, tid in zip(texts, scores, tids):
            if tid < 0:
                continue
            tracks[tid]["frames"].add(int(fnum))
            tracks[tid]["text_scores"][text] += float(score)

    instances = []
    for tid, info in tracks.items():
        frames = sorted(info["frames"])
        best_text = max(info["text_scores"].items(), key=lambda kv: kv[1])[0]
        instances.append({
            "text": best_text,
            "start_sec": round(frames[0] / FPS, 3),
            "end_sec": round((frames[-1] + 1) / FPS, 3),
        })

    # 시작시각 순 정렬 (읽기 편하고 학습에도 일관성 있음)
    instances.sort(key=lambda x: (x["start_sec"], x["end_sec"]))

    result = {
        "video": video_name,
        "instances": instances,
    }
    os.makedirs(os.path.dirname(out_json), exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

'''
def process_one(ocr_pq, seg_pq, out_json, video_name):
    df_ocr = pd.read_parquet(ocr_pq).set_index("frame_num")
    df_seg = pd.read_parquet(seg_pq).set_index("frame_num")

    # track_id → {"frames": set, "text_scores": {text: sum_score}}
    tracks = defaultdict(lambda: {"frames": set(), "text_scores": defaultdict(float)})

    for fnum, ocr_row in df_ocr.iterrows():
        if fnum not in df_seg.index:
            continue
        texts = _loads(ocr_row["ocr_texts"])
        scores = _loads(ocr_row["ocr_scores"])
        tids = _loads(df_seg.loc[fnum, "track_id"])

        for text, score, tid in zip(texts, scores, tids):
            if tid < 0:
                continue
            tracks[tid]["frames"].add(int(fnum))
            tracks[tid]["text_scores"][text] += float(score)

    instances = []
    for tid, info in tracks.items():
        frames = sorted(info["frames"])
        # score 가중 최빈값
        best_text = max(info["text_scores"].items(), key=lambda kv: kv[1])[0]
        instances.append({
            "text": best_text,
            "start_sec": round(frames[0] / FPS, 3),
            "end_sec": round((frames[-1] + 1) / FPS, 3),
        })

    instances, num_tracks = assign_kdenlive_tracks(instances)

    result = {
        "video": video_name,
        "instances": instances,
        "num_tracks": num_tracks,
    }
    os.makedirs(os.path.dirname(out_json), exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
'''
# 배치 실행
tasks = []
for channel in sorted(os.listdir(SEG_DIR)):
    seg_ch = os.path.join(SEG_DIR, channel)
    if not os.path.isdir(seg_ch):
        continue
    for fname in sorted(os.listdir(seg_ch)):
        if not fname.endswith(".parquet"):
            continue
        vn = fname[:-8]
        vn_clean = vn.rsplit("__", 1)[0]   # ← 추가: video 필드용
        seg_pq = os.path.join(seg_ch, fname)
        ocr_pq = os.path.join(OCR_DIR, channel, fname)
        out_json = os.path.join(OUT_DIR, channel, vn + ".json")   # 파일명은 원본 vn
        if not os.path.exists(ocr_pq) or os.path.exists(out_json):
            continue
        tasks.append((ocr_pq, seg_pq, out_json, vn_clean))        # JSON video 필드는 clean
        
print(f"🎯 {len(tasks)}개 처리 예정")

with ProcessPoolExecutor(max_workers=32) as pool:
    futures = [pool.submit(process_one, *t) for t in tasks]
    for f in tqdm(as_completed(futures), total=len(futures)):
        f.result()

🎯 66400개 처리 예정


  0%|          | 0/66400 [00:00<?, ?it/s]